# TerraFlow: forecasting glacier velocity

A Transformer encoder (4 layers, 8 heads, width 256, dropout 0.2, about 5.5 million parameters) that forecasts a pixel's maximum speed on its next observed day from the previous 32 observed days of the same pixel: longitude, latitude, year, month and day as sine and cosine, the daily mean and maximum speed, and the time before the target. Nothing from the target day enters the input. ITS_LIVE observes a pixel on irregular days, so results are reported by forecast horizon.

Split by time: targets up to 2020 train, 2021 chooses the epochs and 2022 onward is the test. Baselines: persistence (the pixel's last observed maximum) and the pixel's typical speed (median of its training days). Training: L1 loss, AdamW with learning rate 1e-4 reduced on plateau, batch 2048, one million samples drawn per epoch, early stopping after 5 epochs without improvement.

**Input:** `Shisper_velocity_ITSLIVE/shisper_velocity.csv.gz` (notebook 05). **Output** (`TerraFlow_Shisper`): test scores, `terraflow_nextday.png`, and the forecasts for every pixel-day used by FusionFlow. GPU runtime (T4), about an hour; training resumes from the last epoch after a disconnect.

## Setup

In [ ]:
import os, json, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
VEL = f'{DRIVE}/Shisper_velocity_ITSLIVE/shisper_velocity.csv.gz'
OUT = f'{DRIVE}/TerraFlow_Shisper'
os.makedirs(OUT, exist_ok=True)

SEQ_LENGTH, D_MODEL, NHEAD, NUM_LAYERS, DROPOUT = 32, 256, 8, 4, 0.2
BATCH_SIZE, LEARNING_RATE = 2048, 1e-4
VSCALE = 100.0                              # speeds in units of 100 m/yr
EPOCH_SAMPLES, VAL_SAMPLES = 1_000_000, 200_000
NUM_EPOCHS, EARLY_STOP = 40, 5
TRAIN_LAST_YEAR, VAL_YEAR = 2020, 2021      # test: 2022 onward
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = device.type == 'cuda'
EPOCH = pd.Timestamp('1970-01-01')
def day_number(s):
    return int((pd.Timestamp(s) - EPOCH).days)
print('device:', device)

## Daily maximum and mean speed of every pixel

In [ ]:
CACHE = f'{OUT}/terraflow_daily.npz'
if os.path.exists(CACHE):
    Z = np.load(CACHE)
    data, DAYS, LANDICE, N_RAW = Z['data'], Z['day'], Z['landice'], int(Z['n_raw'])
else:
    t0, parts, N_RAW = time.time(), [], 0
    for ch in pd.read_csv(VEL, usecols=['lat', 'lon', 'mid_date', 'v [m/yr]', 'landice'], chunksize=4_000_000,
                          dtype={'lat': 'float64', 'lon': 'float64', 'v [m/yr]': 'float64'}):
        ch = ch.dropna(subset=['lat', 'lon', 'mid_date', 'v [m/yr]'])
        N_RAW += len(ch)
        ch['day'] = (pd.to_datetime(ch['mid_date'].astype(str).str[:10], format='%Y-%m-%d') - EPOCH).dt.days.astype('int32')
        parts.append(ch.groupby(['lon', 'lat', 'day'], sort=False).agg(
            vmax=('v [m/yr]', 'max'), vsum=('v [m/yr]', 'sum'), n=('v [m/yr]', 'size'), landice=('landice', 'max')))
        print(f'  read {N_RAW:,} rows | {(time.time() - t0) / 60:.1f} min', flush=True)
    G = pd.concat(parts).groupby(level=[0, 1, 2]).agg(vmax=('vmax', 'max'), vsum=('vsum', 'sum'), n=('n', 'sum'),
                                                       landice=('landice', 'max'))
    del parts
    G = G.reset_index().sort_values(['lon', 'lat', 'day'], kind='stable').reset_index(drop=True)
    d = pd.to_datetime(G.day, unit='D')
    month, dom, dim = d.dt.month.values, d.dt.day.values, d.dt.days_in_month.values
    data = np.column_stack([G.lon.values, G.lat.values, d.dt.year.values,
                            np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12),
                            np.sin(2 * np.pi * dom / dim), np.cos(2 * np.pi * dom / dim),
                            G.vsum.values / G.n.values, G.vmax.values]).astype('float32')
    DAYS, LANDICE = G.day.values.astype('int32'), G.landice.values.astype('int8')
    np.savez(CACHE, data=data, day=DAYS, landice=LANDICE, n_raw=N_RAW)
    del G
N = len(data)
print(f'raw records {N_RAW:,}, pixel-days {N:,}')

## Samples, model and training

In [ ]:
newpix = np.r_[True, (data[1:, 0] != data[:-1, 0]) | (data[1:, 1] != data[:-1, 1])]
PIX = np.cumsum(newpix) - 1
start_of_pix = np.maximum.accumulate(np.where(newpix, np.arange(N), 0))
YEAR = data[:, 2].astype(int)
valid = (np.arange(N) - start_of_pix) >= SEQ_LENGTH             # 32 earlier days of the same pixel exist
J = dict(train=np.nonzero(valid & (YEAR <= TRAIN_LAST_YEAR))[0], val=np.nonzero(valid & (YEAR == VAL_YEAR))[0],
         test=np.nonzero(valid & (YEAR > VAL_YEAR))[0])
HORIZON = np.r_[0, np.diff(DAYS)].astype(int)                   # days between a target and the previous observation
print(f'pixels {PIX.max() + 1:,} | samples: train {len(J["train"]):,}, validation {len(J["val"]):,}, test {len(J["test"]):,}')

tr_rows = YEAR <= TRAIN_LAST_YEAR
MU = data[tr_rows, :3].mean(0); SD = data[tr_rows, :3].std(0) + 1e-6
F = data.copy()
F[:, :3] = (F[:, :3] - MU) / SD
F[:, 7:9] = F[:, 7:9] / VSCALE
F_t = torch.from_numpy(F).to(device)
D_t = torch.from_numpy(DAYS.astype('float32')).to(device)
AR = torch.arange(SEQ_LENGTH, device=device)

def gather(j):
    s = (j - SEQ_LENGTH)[:, None] + AR                       # the 32 previous days of the same pixel
    dt = ((D_t[j][:, None] - D_t[s]) / 30.0)[..., None]      # time before the target, in months
    return torch.cat([F_t[s], dt], -1), F_t[j, 8]            # inputs (batch, 32, 10), target (scaled maximum speed)

class TransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(1, 1000, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder[:, :x.size(1), :]
        return self.fc(self.transformer_encoder(x)[:, -1, :])

def predict(m, js):
    m.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(js), BATCH_SIZE):
            X, _ = gather(torch.from_numpy(js[b:b + BATCH_SIZE]).to(device))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
                out.append(m(X).squeeze(-1).float().cpu().numpy())
    return np.concatenate(out) * VSCALE if out else np.array([])

rng = np.random.default_rng(SEED)
JV = np.sort(rng.choice(J['val'], min(VAL_SAMPLES, len(J['val'])), replace=False))   # fixed validation subset
YV = data[JV, 8]

model = TransformerRegressor(10, D_MODEL, NHEAD, NUM_LAYERS, DROPOUT).to(device)
criterion = nn.L1Loss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)
scaler = torch.amp.GradScaler(device.type, enabled=AMP)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

CKPT, BEST = f'{OUT}/terraflow_next_checkpoint.pt', f'{OUT}/terraflow_next_best.pt'
hist, start, best, since_best = [], 0, float('inf'), 0
if os.path.exists(CKPT):
    c = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(c['model']); optimizer.load_state_dict(c['opt'])
    scheduler.load_state_dict(c['sched']); scaler.load_state_dict(c['scaler'])
    hist, start, best, since_best = c['hist'], c['epoch'] + 1, c['best'], c['since_best']
    print(f'resuming after epoch {start}')

for epoch in range(start, NUM_EPOCHS):
    if since_best >= EARLY_STOP:
        break
    t0 = time.time(); model.train()
    pick = np.random.default_rng(SEED + epoch).choice(J['train'], min(EPOCH_SAMPLES, len(J['train'])), replace=False)
    tr, steps = 0.0, 0
    for b in range(0, len(pick), BATCH_SIZE):
        X, yb = gather(torch.from_numpy(pick[b:b + BATCH_SIZE]).to(device))
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
            loss = criterion(model(X).squeeze(-1), yb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr += loss.item(); steps += 1
    val_mae = float(np.abs(predict(model, JV) - YV).mean())
    scheduler.step(val_mae)
    if val_mae < best:
        best, since_best = val_mae, 0
        torch.save(model.state_dict(), BEST)
    else:
        since_best += 1
    hist.append(dict(epoch=epoch + 1, train_mae=tr / steps * VSCALE, val_mae=val_mae, lr=optimizer.param_groups[0]['lr'],
                     minutes=round((time.time() - t0) / 60, 1)))
    torch.save(dict(model=model.state_dict(), opt=optimizer.state_dict(), sched=scheduler.state_dict(),
                    scaler=scaler.state_dict(), hist=hist, epoch=epoch, best=best, since_best=since_best), CKPT)
    print(f"epoch {epoch + 1}: train MAE {tr / steps * VSCALE:.1f} | validation MAE {val_mae:.1f} m/yr", flush=True)

## Test, 2022 onward

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=device))
jt = J['test']
p = predict(model, jt)
y = data[jt, 8]
persist = data[jt - 1, 8]                                       # the pixel's last observed daily maximum
med = pd.Series(data[tr_rows, 8]).groupby(PIX[tr_rows]).median()
typical = pd.Series(PIX[jt]).map(med).fillna(float(np.median(data[tr_rows, 8]))).values
h, glacier = HORIZON[jt], LANDICE[jt] == 1
def mae(a, m):
    return round(float(np.abs(a[m] - y[m]).mean()), 1) if m.any() else None
season = (DAYS[jt] >= day_number('2022-04-01')) & (DAYS[jt] <= day_number('2022-06-30'))
classes = {'all': np.ones(len(y), bool), 'next day': h == 1, '2 to 5 days ahead': (h >= 2) & (h <= 5),
           '6 to 16 days ahead': (h >= 6) & (h <= 16), 'more than 16 days ahead': h > 16,
           'glacier outline': glacier, 'margin': ~glacier, 'observed over 300 m/yr': y > 300,
           'observed over 1000 m/yr': y > 1000, 'April to June 2022': season}
R = pd.DataFrame({k: dict(n=int(m.sum()), terraflow=mae(p, m), persistence=mae(persist, m), typical_speed=mae(typical, m))
                  for k, m in classes.items()}).T
res = dict(test_mae=mae(p, classes['all']), next_day_mae=mae(p, classes['next day']),
           persistence_mae=mae(persist, classes['all']), epochs_run=len(hist),
           best_val_mae=round(min(x['val_mae'] for x in hist), 1), raw_records=N_RAW, pixel_days=int(N),
           samples=dict(train=len(J['train']), val=len(J['val']), test=len(jt)))
json.dump(dict(res, by_class=R.to_dict()), open(f'{OUT}/terraflow_nextday_results.json', 'w'), indent=1)
pd.DataFrame(hist).to_csv(f'{OUT}/terraflow_nextday_history.csv', index=False)
print(res)
print(R.to_string())

# training curve, and 2022 at the test pixel with the most fast observations
T_ = pd.DataFrame(dict(pix=PIX[jt], day=DAYS[jt], y=y, p=p, persist=persist))
fast = T_[T_.y > 300].pix.value_counts()
px = fast.index[0] if len(fast) else T_.pix.value_counts().index[0]
S_ = T_[(T_.pix == px) & (T_.day >= day_number('2022-01-01')) & (T_.day <= day_number('2022-12-31'))]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={'width_ratios': [1, 2]})
H = pd.DataFrame(hist)
ax[0].plot(H.epoch, H.train_mae, label='train'); ax[0].plot(H.epoch, H.val_mae, label='validation (2021)')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('MAE (m/yr)'); ax[0].legend()
dd = pd.to_datetime(S_.day, unit='D')
ax[1].plot(dd, S_.y, '.', ms=5, c='k', label='observed daily maximum')
ax[1].plot(dd, S_.p, '-', lw=1, c='#c0392b', label='TerraFlow forecast')
ax[1].plot(dd, S_.persist, '-', lw=.8, c='0.6', label='persistence')
ax[1].axvline(pd.Timestamp('2022-05-07'), ls='--', c='#2c7bb6', lw=1, label='outburst, 7 May 2022')
lon_, lat_ = data[np.nonzero(PIX == px)[0][0], :2]
ax[1].set_title(f'2022 at the test pixel with most fast days ({lat_:.4f} N, {lon_:.4f} E)', fontsize=10)
ax[1].set_ylabel('m/yr'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/terraflow_nextday.png', dpi=130); plt.show()

## Forecasts for FusionFlow

In [ ]:
# TerraFlow never sees outburst labels, so its forecasts can be used for every year
jall = np.nonzero(valid)[0]
pall = predict(model, jall).astype('float32')
yall = data[jall, 8]
typ_all = pd.Series(PIX[jall]).map(med).fillna(float(np.median(data[tr_rows, 8]))).values.astype('float32')
np.savez_compressed(f'{OUT}/terraflow_forecasts_all.npz', pix=PIX[jall].astype('int32'), day=DAYS[jall], observed=yall,
                    forecast=pall, typical=typ_all, landice=LANDICE[jall])
A = pd.DataFrame(dict(day=DAYS[jall], obs=yall, fc=pall, typ=typ_all))
A['resid'] = A.obs - A.fc
A['ratio'] = A.obs / np.maximum(A.typ, 1.0)
A['fast'] = A.obs > 300
DAILY = A.groupby('day').agg(n_pixels=('obs', 'size'), median_observed=('obs', 'median'), median_forecast=('fc', 'median'),
                             median_resid=('resid', 'median'), median_ratio_to_typical=('ratio', 'median'),
                             frac_over_300=('fast', 'mean')).reset_index()
DAILY.insert(0, 'date', pd.to_datetime(DAILY.day, unit='D').dt.strftime('%Y-%m-%d'))
DAILY.round(4).to_csv(f'{OUT}/terraflow_daily_for_fusion.csv', index=False)
print(f'forecasts for {len(jall):,} pixel-days saved | daily summary {DAILY.date.min()} to {DAILY.date.max()}')